In [1]:
# ============================================================
# CHECKPOINT 1 — FRAMING THE PROBLEM
# ============================================================

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)

# Load dataset
df = pd.read_csv("house_sale.csv")

# ------------------------------------------------------------
# 1. Business Problem / Target Validation
# ------------------------------------------------------------

print("Shape:", df.shape)

print("\nCurrency distribution:")
print(df["currency_x"].value_counts(dropna=False))

print(
    "\nRaw price range (AZN):",
    df["price"].min(),
    "-",
    df["price"].max()
)

# Target checks
assert "price" in df.columns, "Target column 'price' not found"
assert pd.api.types.is_numeric_dtype(
    df["price"]
), "price is not numeric"
assert df["price"].isnull().sum() == 0, (
    "price has missing values"
)
assert (df["price"] > 0).all(), (
    "Found non-positive price values"
)

print("\nTarget column:", "price")
print("Rows:", len(df))
print(
    "Price range (AZN):",
    df["price"].min(),
    "-",
    df["price"].max()
)
print("All initial target checks passed.")


# ============================================================
# CHECKPOINT 2 — FULL EDA AND CLEANING
# ============================================================

# ------------------------------------------------------------
# 2.1 Initial Dataset Inspection
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2.1 INITIAL DATASET INSPECTION")
print("=" * 70)

print("\nShape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nExact duplicate rows:")
print(df.duplicated().sum())

print("\nFirst 5 rows:")
display(df.head())

print("\nNumerical summary:")
display(df.describe())


# ------------------------------------------------------------
# 2.2 Checking price and total_price
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2.2 PRICE VS TOTAL_PRICE")
print("=" * 70)

price_equal_rate = (
    df["price"] == df["total_price"]
).mean()

print(
    "Percentage of rows where price == total_price:",
    price_equal_rate
)

print("\nDifference between price and total_price:")
display(
    (df["price"] - df["total_price"]).describe()
)


# ------------------------------------------------------------
# 2.3 Checking _x and _y Columns
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2.3 _X / _Y COLUMN CHECK")
print("=" * 70)

pairs = [
    ("id_x", "id_y"),
    ("estate_rel_url_x", "estate_rel_url_y"),
    ("datetime_scrape_x", "datetime_scrape_y"),
    ("currency_x", "currency_y"),
    ("estate_details_id_x", "estate_details_id_y"),
    ("estate_rel_url_x", "estate_rel_url"),
]

for col1, col2 in pairs:
    same = (
        df[col1].fillna("__MISSING__")
        == df[col2].fillna("__MISSING__")
    ).mean()

    print(
        f"{col1} vs {col2}: {same:.2%} identical"
    )


# ------------------------------------------------------------
# 2.4 Investigating Extreme Price Values
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2.4 EXTREME PRICE VALUES")
print("=" * 70)

print("Prices below 1,000 AZN:")
display(
    df[df["price"] < 1000][
        [
            "price",
            "location",
            "attributes",
            "Sahə",
            "Otaq sayı"
        ]
    ].head(20)
)

print("\nPrices above 10 million AZN:")
display(
    df[df["price"] > 10_000_000][
        [
            "price",
            "location",
            "attributes",
            "Sahə",
            "Otaq sayı"
        ]
    ].head(20)
)

print(
    "\nBelow 1,000 AZN:",
    (df["price"] < 1000).sum()
)

print(
    "Above 10 million AZN:",
    (df["price"] > 10_000_000).sum()
)


# ------------------------------------------------------------
# 2.5 Repeated Property Listings
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2.5 REPEATED PROPERTY LISTINGS")
print("=" * 70)

print(
    "Duplicate estate IDs:",
    df["estate_id"].duplicated().sum()
)

print(
    "Duplicate property URLs:",
    df["estate_rel_url"].duplicated().sum()
)

url = (
    df["estate_rel_url"]
    .value_counts()
    .idxmax()
)

print("\nMost repeated property URL:")
print(url)

display(
    df[df["estate_rel_url"] == url][
        [
            "estate_rel_url",
            "estate_id",
            "price",
            "Sahə",
            "Otaq sayı",
            "location"
        ]
    ]
)

print("\nScraping dates for repeated property:")
display(
    df[df["estate_rel_url"] == url][
        [
            "estate_rel_url",
            "estate_id",
            "price",
            "datetime_scrape_x",
            "datetime_scrape_y"
        ]
    ]
)

url_counts = df["estate_rel_url"].value_counts()

print(
    "\nProperties appearing more than once:",
    (url_counts > 1).sum()
)

print(
    "Maximum appearances of one property:",
    url_counts.max()
)

price_changes = (
    df.groupby("estate_rel_url")["price"]
    .nunique()
)

print(
    "\nProperties with different prices:",
    (price_changes > 1).sum()
)

print(
    "Properties with the same price:",
    (price_changes == 1).sum()
)

repeated_rows = (
    df["estate_rel_url"]
    .duplicated(keep=False)
    .sum()
)

print(
    "\nRows belonging to repeated properties:",
    repeated_rows
)

print(
    "Unique properties:",
    df["estate_rel_url"].nunique()
)

print(
    "Rows if one row per property were kept:",
    df.drop_duplicates("estate_rel_url").shape[0]
)


# ------------------------------------------------------------
# 2.6 Checking unit_price
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2.6 UNIT_PRICE")
print("=" * 70)

print(
    df["unit_price"]
    .dropna()
    .head(20)
    .tolist()
)

print("\nRandom examples:")
print(
    df["unit_price"]
    .dropna()
    .sample(
        20,
        random_state=42
    )
    .tolist()
)


# ------------------------------------------------------------
# 2.7 Create Working Modeling DataFrame
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2.7 CREATE WORKING DATAFRAME")
print("=" * 70)

drop_cols = [
    "total_price",
    "unit_price",
    "estate_rel_url_y",
    "currency_y",
    "estate_details_id_y",
    "datetime_scrape_y",
    "id_x",
    "id_y",
    "estate_id",
    "estate_details_id_x",
    "rel_url",
    "img_url"
]

df_model = df.drop(
    columns=drop_cols,
    errors="ignore"
).copy()

print(
    "Shape after removing leakage/redundant identifier columns:",
    df_model.shape
)

print("\nRemaining columns:")
print(df_model.columns.tolist())


# ------------------------------------------------------------
# 2.8 Missing Value Analysis
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2.8 MISSING VALUE ANALYSIS")
print("=" * 70)

missing_percent = (
    df_model.isnull().mean() * 100
).sort_values(ascending=False)

print("Missing values (%):")
display(missing_percent)


# ------------------------------------------------------------
# 2.9 Remove Features with >80% Missing Values
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2.9 HIGH MISSINGNESS FEATURES")
print("=" * 70)

high_missing_cols = (
    missing_percent[
        missing_percent > 80
    ]
    .index
    .tolist()
)

print(
    "Features with more than 80% missing values:"
)
print(high_missing_cols)

df_model = df_model.drop(
    columns=high_missing_cols
)

print(
    "\nShape after removing highly incomplete features:",
    df_model.shape
)


# ------------------------------------------------------------
# 2.10 Checking Duplicate Categorical Features
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2.10 DUPLICATE CATEGORICAL FEATURES")
print("=" * 70)

# repair vs Təmir
repair_comparison = pd.DataFrame({
    "repair": df_model["repair"].fillna(
        "__MISSING__"
    ),
    "Təmir": df_model["Təmir"].fillna(
        "__MISSING__"
    )
})

print(
    "repair vs Təmir:",
    (
        repair_comparison["repair"]
        == repair_comparison["Təmir"]
    ).mean()
)

print("\nrepair vs Təmir crosstab:")
display(
    pd.crosstab(
        df_model["repair"].fillna("__MISSING__"),
        df_model["Təmir"].fillna("__MISSING__"),
        dropna=False
    )
)

# mortgage vs İpoteka
mortgage_comparison = pd.DataFrame({
    "mortgage": df_model["mortgage"].fillna(
        "__MISSING__"
    ),
    "İpoteka": df_model["İpoteka"].fillna(
        "__MISSING__"
    )
})

print(
    "\nmortgage vs İpoteka:",
    (
        mortgage_comparison["mortgage"]
        == mortgage_comparison["İpoteka"]
    ).mean()
)

print("\nmortgage vs İpoteka crosstab:")
display(
    pd.crosstab(
        df_model["mortgage"].fillna("__MISSING__"),
        df_model["İpoteka"].fillna("__MISSING__"),
        dropna=False
    )
)

# bill_of_sale vs Çıxarış
bill_comparison = pd.DataFrame({
    "bill_of_sale": df_model["bill_of_sale"].fillna(
        "__MISSING__"
    ),
    "Çıxarış": df_model["Çıxarış"].fillna(
        "__MISSING__"
    )
})

print(
    "\nbill_of_sale vs Çıxarış:",
    (
        bill_comparison["bill_of_sale"]
        == bill_comparison["Çıxarış"]
    ).mean()
)

print("\nbill_of_sale vs Çıxarış crosstab:")
display(
    pd.crosstab(
        df_model["bill_of_sale"].fillna("__MISSING__"),
        df_model["Çıxarış"].fillna("__MISSING__"),
        dropna=False
    )
)

duplicate_categorical_cols = [
    "repair",
    "mortgage",
    "bill_of_sale"
]

df_model = df_model.drop(
    columns=duplicate_categorical_cols
)

print(
    "\nShape after removing redundant categorical columns:",
    df_model.shape
)


# ------------------------------------------------------------
# 2.11 Re-check Missing Values
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2.11 REMAINING MISSING VALUES")
print("=" * 70)

missing_pct = (
    df_model.isnull().mean() * 100
).sort_values(ascending=False)

display(
    missing_pct[
        missing_pct > 0
    ]
)


# ------------------------------------------------------------
# 2.12 Inspect Remaining Missing Values
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2.12 INSPECT REMAINING MISSING VALUES")
print("=" * 70)

missing_cols = df_model.columns[
    df_model.isnull().any()
]

for col in missing_cols:
    print(f"\n===== {col} =====")
    display(
        df_model[col]
        .value_counts(
            dropna=False
        )
        .head(10)
    )


# ------------------------------------------------------------
# 2.13 Area Unit Analysis
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2.13 AREA UNIT ANALYSIS")
print("=" * 70)

df_model["area_unit"] = (
    df_model["Sahə"]
    .str.extract(
        r"(m²|m2|sot)",
        expand=False
    )
)

print(
    df_model["area_unit"]
    .value_counts(
        dropna=False
    )
)


# ------------------------------------------------------------
# 2.14 Investigating Missing Room Counts
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2.14 MISSING ROOM COUNTS")
print("=" * 70)

print(
    "Room counts:"
)

display(
    df_model["Otaq sayı"]
    .value_counts(
        dropna=False
    )
    .sort_index()
)

missing_rooms_by_unit = (
    df_model
    .groupby("area_unit")["Otaq sayı"]
    .apply(
        lambda x: x.isna().sum()
    )
)

print(
    "\nMissing room counts by area unit:"
)

display(
    missing_rooms_by_unit
)

print(
    "\nMissing room count for m² listings:"
)

display(
    df_model[
        df_model["Otaq sayı"].isna()
        &
        (df_model["area_unit"] == "m²")
    ][
        [
            "price",
            "Sahə",
            "location",
            "Təmir",
            "İpoteka"
        ]
    ].head(30)
)


# ------------------------------------------------------------
# 2.15 Convert Area to Numerical Features
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2.15 AREA NUMERICAL FEATURES")
print("=" * 70)

df_model["area_value"] = (
    df_model["Sahə"]
    .str.extract(
        r"([\d.]+)",
        expand=False
    )
    .astype(float)
)

df_model["area_m2"] = (
    df_model["area_value"]
)

df_model.loc[
    df_model["area_unit"] == "sot",
    "area_m2"
] = (
    df_model.loc[
        df_model["area_unit"] == "sot",
        "area_value"
    ] * 100
)

display(
    df_model[
        [
            "Sahə",
            "area_unit",
            "area_value",
            "area_m2"
        ]
    ].head(20)
)


# ------------------------------------------------------------
# 2.16 Missing Categorical Value Analysis
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2.16 MISSING CATEGORICAL VALUES")
print("=" * 70)

cols_to_check = [
    "İpoteka",
    "Təmir",
    "Otaq sayı",
    "products_label"
]

for col in cols_to_check:
    print(f"\n===== {col} =====")

    print(
        "Missing:",
        df_model[col].isna().sum()
    )

    print(
        "Missing %:",
        round(
            df_model[col].isna().mean() * 100,
            2
        )
    )


# ------------------------------------------------------------
# 2.17 Target Distribution by Missingness
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2.17 TARGET DISTRIBUTION BY MISSINGNESS")
print("=" * 70)

for col in cols_to_check:

    print(f"\n===== {col} =====")

    print(
        "Price when value is missing:"
    )

    display(
        df_model.loc[
            df_model[col].isna(),
            "price"
        ].describe()
    )

    print(
        "Price when value is not missing:"
    )

    display(
        df_model.loc[
            df_model[col].notna(),
            "price"
        ].describe()
    )


# ------------------------------------------------------------
# 2.18 Investigating Mortgage
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2.18 MORTGAGE ANALYSIS")
print("=" * 70)

print(
    df_model["İpoteka"]
    .value_counts(
        dropna=False
    )
)

print("\nMortgage = var:")
display(
    df_model[
        df_model["İpoteka"] == "var"
    ][
        [
            "price",
            "Sahə",
            "location",
            "Otaq sayı",
            "Təmir"
        ]
    ].head(20)
)

print("\nMortgage = missing:")
display(
    df_model[
        df_model["İpoteka"].isna()
    ][
        [
            "price",
            "Sahə",
            "location",
            "Otaq sayı",
            "Təmir"
        ]
    ].head(20)
)


# ------------------------------------------------------------
# 2.19 Investigating products_label
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2.19 PRODUCTS_LABEL ANALYSIS")
print("=" * 70)

print(
    df_model["products_label"]
    .value_counts(
        dropna=False
    )
)

print("\nProducts_label = missing:")
display(
    df_model[
        df_model["products_label"].isna()
    ][
        [
            "price",
            "Sahə",
            "location",
            "Otaq sayı",
            "Təmir",
            "İpoteka"
        ]
    ].head(20)
)

print("\nProducts_label = available:")
display(
    df_model[
        df_model["products_label"].notna()
    ][
        [
            "price",
            "Sahə",
            "location",
            "Otaq sayı",
            "Təmir",
            "İpoteka"
        ]
    ].head(20)
)

print(
    "\nPrice when products_label is missing:"
)

display(
    df_model.loc[
        df_model["products_label"].isna(),
        "price"
    ].describe()
)

print(
    "\nPrice when products_label is not missing:"
)

display(
    df_model.loc[
        df_model["products_label"].notna(),
        "price"
    ].describe()
)

print(
    "\nPrice distribution by products_label:"
)

display(
    df_model.groupby(
        df_model["products_label"]
        .fillna("Missing")
    )["price"].describe()
)

for col in [
    "location",
    "Təmir",
    "İpoteka"
]:

    print(
        f"\n===== {col} vs products_label ====="
    )

    display(
        pd.crosstab(
            df_model[col],
            df_model["products_label"],
            normalize="index"
        ).round(3)
    )


# ------------------------------------------------------------
# 2.20 Fill Categorical Missing Values
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2.20 CATEGORICAL MISSING VALUE HANDLING")
print("=" * 70)

for col in [
    "İpoteka",
    "Təmir",
    "Çıxarış",
    "products_label"
]:

    df_model[col] = (
        df_model[col]
        .fillna("Missing")
    )

print("\nİpoteka:")
print(
    df_model["İpoteka"]
    .value_counts()
)

print("\nTəmir:")
print(
    df_model["Təmir"]
    .value_counts()
)

print("\nÇıxarış:")
print(
    df_model["Çıxarış"]
    .value_counts()
)

print("\nproducts_label:")
print(
    df_model["products_label"]
    .value_counts()
)


# ------------------------------------------------------------
# 2.21 Seller / Agency Features
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2.21 SELLER / AGENCY FEATURES")
print("=" * 70)

print(
    df_model[
        [
            "owner_name",
            "owner_title",
            "shop_name",
            "shop_title"
        ]
    ].isnull().sum()
)

seller_cols = [
    "owner_name",
    "shop_name",
    "shop_title"
]

df_model = df_model.drop(
    columns=seller_cols
)

print(
    "Shape after removing seller/agency identifier-like columns:",
    df_model.shape
)


# ------------------------------------------------------------
# 2.22 Floor Features
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2.22 FLOOR FEATURES")
print("=" * 70)

print(
    "Unique Mərtəbə values:",
    df_model["Mərtəbə"].nunique()
)

display(
    df_model["Mərtəbə"]
    .dropna()
    .head(20)
)

floor_parts = (
    df_model["Mərtəbə"]
    .str.split(
        "/",
        expand=True
    )
)

df_model["floor"] = pd.to_numeric(
    floor_parts[0],
    errors="coerce"
)

df_model["total_floors"] = pd.to_numeric(
    floor_parts[1],
    errors="coerce"
)

display(
    df_model[
        [
            "Mərtəbə",
            "floor",
            "total_floors"
        ]
    ].head(20)
)


# ------------------------------------------------------------
# 2.23 Price per m² — Exploratory Analysis Only
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2.23 PRICE PER M²")
print("=" * 70)

df_model["price_per_m2"] = np.where(
    df_model["area_m2"] > 0,
    df_model["price"] /
    df_model["area_m2"],
    np.nan
)

display(
    df_model[
        [
            "price",
            "Sahə",
            "area_m2",
            "price_per_m2"
        ]
    ].head(20)
)

display(
    df_model["price_per_m2"].describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)


# ------------------------------------------------------------
# 2.24 Extreme Price per m² Values
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2.24 EXTREME PRICE PER M² VALUES")
print("=" * 70)

print("Highest price/m² observations:")

display(
    df_model.nlargest(
        20,
        "price_per_m2"
    )[
        [
            "price",
            "Sahə",
            "area_unit",
            "area_m2",
            "price_per_m2",
            "location"
        ]
    ]
)

print("Lowest price/m² observations:")

display(
    df_model.nsmallest(
        20,
        "price_per_m2"
    )[
        [
            "price",
            "Sahə",
            "area_unit",
            "area_m2",
            "price_per_m2",
            "location"
        ]
    ]
)


# ------------------------------------------------------------
# 2.25 Flag Potentially Unusual Price per m²
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2.25 PRICE PER M² FLAGS")
print("=" * 70)

df_model["price_per_m2_flag"] = (
    (df_model["price_per_m2"] < 100)
    |
    (df_model["price_per_m2"] > 10000)
)

print(
    "Potentially unusual price/m² observations:",
    df_model["price_per_m2_flag"].sum()
)

display(
    df_model[
        df_model["price_per_m2_flag"]
    ][
        [
            "price",
            "Sahə",
            "area_unit",
            "area_value",
            "area_m2",
            "price_per_m2",
            "location"
        ]
    ]
    .sort_values(
        "price_per_m2"
    )
    .head(20)
)


# ------------------------------------------------------------
# 2.26 Extremely Large Areas
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2.26 EXTREMELY LARGE AREAS")
print("=" * 70)

display(
    df_model["area_m2"].describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)

large_area_mask = (
    df_model["area_m2"] > 10000
)

print(
    "Properties with area > 10,000 m²:",
    large_area_mask.sum()
)

display(
    df_model[
        large_area_mask
    ][
        [
            "price",
            "Sahə",
            "area_unit",
            "area_value",
            "area_m2",
            "location"
        ]
    ]
    .sort_values(
        "area_m2",
        ascending=False
    )
    .head(30)
)


# ------------------------------------------------------------
# 2.27 Remove Temporary EDA Variables
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2.27 REMOVE TEMPORARY EDA VARIABLES")
print("=" * 70)

df_model = df_model.drop(
    columns=[
        "price_per_m2",
        "price_per_m2_flag"
    ],
    errors="ignore"
)


# ------------------------------------------------------------
# 2.28 Remove Original Text Area / Floor Columns
# ------------------------------------------------------------

df_model = df_model.drop(
    columns=[
        "Sahə",
        "Mərtəbə"
    ],
    errors="ignore"
)


# ------------------------------------------------------------
# 2.29 Final Feature Cleanup
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2.29 FINAL FEATURE CLEANUP")
print("=" * 70)

# Constant currency feature
if (
    "currency_x" in df_model.columns
    and df_model["currency_x"].nunique(dropna=False) == 1
):
    df_model = df_model.drop(
        columns=["currency_x"]
    )

# Remove redundant URL copy if it is identical
if (
    "estate_rel_url_x" in df_model.columns
    and "estate_rel_url" in df_model.columns
):
    url_same = (
        df_model["estate_rel_url_x"].fillna("__MISSING__")
        ==
        df_model["estate_rel_url"].fillna("__MISSING__")
    ).mean()

    print(
        "estate_rel_url_x vs estate_rel_url:",
        f"{url_same:.2%}"
    )

    if url_same == 1.0:
        df_model = df_model.drop(
            columns=["estate_rel_url_x"]
        )


# ------------------------------------------------------------
# 2.30 Final Dataset Check
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2.30 FINAL DATASET CHECK")
print("=" * 70)

print(
    "Final cleaned dataframe shape:",
    df_model.shape
)

print("\nFinal columns:")
print(df_model.columns.tolist())

print("\nRemaining missing values:")
display(
    df_model.isnull()
    .sum()
    .sort_values(
        ascending=False
    )
    .head(30)
)

print("\nData types:")
print(df_model.dtypes)


# ------------------------------------------------------------
# 2.31 Final Leakage / Identifier Check
# ------------------------------------------------------------

leakage_or_identifier_cols = [
    "total_price",
    "unit_price",
    "id_x",
    "id_y",
    "estate_id",
    "estate_rel_url_x",
    "estate_rel_url_y",
    "estate_details_id_x",
    "estate_details_id_y",
    "rel_url",
    "img_url",
    "price_per_m2",
    "price_per_m2_flag"
]

remaining_leakage_cols = [
    col
    for col in leakage_or_identifier_cols
    if col in df_model.columns
]

print(
    "\nRemaining leakage/identifier columns:",
    remaining_leakage_cols
)


# ------------------------------------------------------------
# 2.32 Group Identifier for Repeated Properties
# ------------------------------------------------------------

if "estate_rel_url" in df_model.columns:

    groups = (
        df_model["estate_rel_url"]
        .copy()
    )

    print(
        "\nUnique property groups:",
        groups.nunique()
    )

    print(
        "Total observations:",
        len(groups)
    )


# ------------------------------------------------------------
# 2.33 Final Target / Feature Separation
# ------------------------------------------------------------

y = df_model["price"].copy()

X = df_model.drop(
    columns=["price"]
).copy()

print(
    "\nX shape:",
    X.shape
)

print(
    "y shape:",
    y.shape
)

print(
    "\nTarget range:",
    y.min(),
    "-",
    y.max()
)


# ------------------------------------------------------------
# 2.34 Final Assertions
# ------------------------------------------------------------

assert "price" not in X.columns

assert (
    "price_per_m2" not in X.columns
)

assert (
    "price_per_m2_flag" not in X.columns
)

assert (
    "total_price" not in X.columns
)

assert (
    "unit_price" not in X.columns
)

assert y.isnull().sum() == 0

assert (y > 0).all()

print(
    "\nAll final Checkpoint 2 checks passed."
)

Shape: (100775, 51)

Currency distribution:
currency_x
AZN    100775
Name: count, dtype: int64

Raw price range (AZN): 11.0 - 600000000.0

Target column: price
Rows: 100775
Price range (AZN): 11.0 - 600000000.0
All initial target checks passed.

2.1 INITIAL DATASET INSPECTION

Shape:
(100775, 51)

Columns:
['id_x', 'rel_url', 'estate_rel_url_x', 'datetime_scrape_x', 'price', 'currency_x', 'location', 'attributes', 'city_when', 'city', 'day_x', 'hour_x', 'repair', 'vip', 'featured', 'products_label', 'bill_of_sale', 'mortgage', 'img_url', 'id_y', 'estate_id', 'estate_rel_url_y', 'datetime_scrape_y', 'description', 'unit_price', 'total_price', 'currency_y', 'owner_name', 'owner_title', 'shop_name', 'shop_title', 'address', 'lat', 'lng', 'updated', 'views', 'day_y', 'hour_y', 'estate_details_id_x', 'Binanın növü', 'Kateqoriya', 'Mərtəbə', 'Otaq sayı', 'Sahə', 'Torpaq sahəsi', 'Təmir', 'Çıxarış', 'İpoteka', 'estate_details_id_y', 'estate_rel_url', 'extra_info']

Data types:
id_x           

,id_x,rel_url,estate_rel_url_x,datetime_scrape_x,price,currency_x,location,attributes,city_when,city,day_x,hour_x,repair,vip,featured,products_label,bill_of_sale,mortgage,img_url,id_y,estate_id,estate_rel_url_y,datetime_scrape_y,description,unit_price,total_price,currency_y,owner_name,owner_title,shop_name,shop_title,address,lat,lng,updated,views,day_y,hour_y,estate_details_id_x,Binanın növü,Kateqoriya,Mərtəbə,Otaq sayı,Sahə,Torpaq sahəsi,Təmir,Çıxarış,İpoteka,estate_details_id_y,estate_rel_url,extra_info
0,5df36281-6dc6-4d5d-89a7-5fcfa86f7608,/alqi-satqi?page=174,/items/4521724,2024-10-05 22:07:37.60613+00,499999.0,AZN,Səbail r.,"4 otaqlı, 145 m², 7/9 mərtəbə","Bakı, dünən 23:52",bakı,05.10.2024,23:52,Təmirli,vipped,featured,NaN,Çıxarış var,NaN,https://bina.azstatic.com/uploads/f460x345/202...,92e82ea2-e1f3-4c2d-a284-151efd99281e,5df36281-6dc6-4d5d-89a7-5fcfa86f7608,/items/4521724,2024-10-05 22:14:10.089116+00,"Səbail Rayonu, İzzət Nəbiyev küçəsi, Fəxri Xiy...",3 450 AZN/m²,499999.0,AZN,Kamran,mülkiyyətçi,NaN,NaN,İzzət Nəbiyev küç.,40.358817,49.824092,yeniləndi: dünən 23:52,1155,05.10.2024,23:52,92e82ea2-e1f3-4c2d-a284-151efd99281e,NaN,Köhnə tikili,7 / 9,4.0,145 m²,NaN,var,var,NaN,92e82ea2-e1f3-4c2d-a284-151efd99281e,/items/4521724,Şəhidlər xiyabanı * Dağüstü parkı * Səbail r.
1,883e20f0-8872-49a5-8b4b-63b8301b5f8f,/alqi-satqi?page=250,/items/4669294,2024-10-05 22:07:37.60613+00,77000.0,AZN,Biləcəri q.,"4 otaqlı, 90 m²","Bakı, dünən 23:56",bakı,05.10.2024,23:56,Təmirli,NaN,NaN,NaN,NaN,NaN,https://bina.azstatic.com/uploads/f460x345/202...,505eaf81-6bc8-4094-9b00-aa82066548ee,883e20f0-8872-49a5-8b4b-63b8301b5f8f,/items/4669294,2024-10-05 22:14:10.089116+00,"Biləcəridə Abidəyə yaxin 91,92,202 saylı marşr...",NaN,77000.0,AZN,Dasinmaz Emlak,vasitəçi (agent),NaN,NaN,Biləcəri qəs.,40.420897,49.807035,yeniləndi: 04 oktyabr 2024,218,04.10.2024,NaN,505eaf81-6bc8-4094-9b00-aa82066548ee,NaN,Həyət evi/Bağ evi,NaN,4.0,90 m²,1.3 sot,var,yoxdur,NaN,505eaf81-6bc8-4094-9b00-aa82066548ee,/items/4669294,Binəqədi r.* Biləcəri q.
2,55c36fb1-a3af-476e-ba17-a81f6795be8d,/alqi-satqi?page=250,/items/4669293,2024-10-05 22:07:37.60613+00,92000.0,AZN,İnşaatçılar m.,"3 otaqlı, 60 m²","Bakı, dünən 23:55",bakı,05.10.2024,23:55,Təmirli,NaN,NaN,NaN,Çıxarış var,NaN,https://bina.azstatic.com/uploads/f460x345/202...,fa63b201-999d-43b5-a61a-778d9d79a6c6,55c36fb1-a3af-476e-ba17-a81f6795be8d,/items/4669293,2024-10-05 22:14:10.089116+00,Salam əleykum. \nİnşaatçılar metrosuna yaxın m...,NaN,92000.0,AZN,Məhəmməd,vasitəçi (agent),NaN,NaN,Mirzə Cabbar Məmmədzadə küç.,40.390293,49.802656,yeniləndi: 04 oktyabr 2024,190,04.10.2024,NaN,fa63b201-999d-43b5-a61a-778d9d79a6c6,NaN,Həyət evi/Bağ evi,NaN,3.0,60 m²,0.1 sot,var,var,NaN,fa63b201-999d-43b5-a61a-778d9d79a6c6,/items/4669293,İnşaatçılar m.* Yasamal r.
3,acf1aa8d-a46a-40f5-b6f2-f7a449569337,/alqi-satqi?page=250,/items/4647811,2024-10-05 22:07:37.60613+00,95000.0,AZN,Qaraçuxur q.,130 m²,"Bakı, dünən 23:55",bakı,05.10.2024,23:55,Təmirli,vipped,featured,NaN,Çıxarış var,İpoteka var,https://bina.azstatic.com/uploads/f460x345/202...,5db56980-05cc-4925-b55b-f58fb3f4d2b6,acf1aa8d-a46a-40f5-b6f2-f7a449569337,/items/4647811,2024-10-05 22:14:10.089116+00,Barter maraqlidir üstünlük maşina verilir\nHər...,NaN,95000.0,AZN,Elçin,mülkiyyətçi,NaN,NaN,Qaraçuxur qəs.,40.393614,49.981553,yeniləndi: 04 oktyabr 2024,1314,04.10.2024,NaN,5db56980-05cc-4925-b55b-f58fb3f4d2b6,NaN,Obyekt,NaN,NaN,130 m²,NaN,var,var,var,5db56980-05cc-4925-b55b-f58fb3f4d2b6,/items/4647811,Suraxanı r.* Qaraçuxur q.
4,22d840df-9283-4112-bc71-7432511fc776,/alqi-satqi?page=250,/items/4638863,2024-10-05 22:07:37.60613+00,220000.0,AZN,Əhmədli m.,"3 otaqlı, 100 m², 15/16 mərtəbə","Bakı, dünən 23:52",bakı,05.10.2024,23:52,Təmirli,NaN,NaN,Agentlik,Çıxarış var,NaN,https://bina.azstatic.com/uploads/f460x345/202...,d71cf9a9-86dd-4162-b540-6252a8659a09,22d840df-9283-4112-bc71-7432511fc776,/items/4638863,2024-10-05 22:14:10.089116+00,Əhmədli qəs. Qaçaq Nəbi küçəsi 3 otaga duzelm...


Numerical summary:


,price,total_price,lat,lng,views,Otaq sayı
count,1.007750e+05,1.007750e+05,100775.000000,100775.000000,100775.000000,91363.000000
mean,3.423557e+05,3.423553e+05,40.410846,49.886371,700.065185,3.133468
std,2.042627e+06,2.042627e+06,0.083193,0.265008,1680.672574,1.372981
min,1.100000e+01,1.100000e+01,32.689217,12.591688,23.000000,1.000000
25%,1.450000e+05,1.450000e+05,40.381900,49.816470,99.000000,2.000000
50%,2.180000e+05,2.180000e+05,40.397219,49.852867,254.000000,3.000000
75%,3.380000e+05,3.380000e+05,40.420578,49.948995,680.000000,4.000000
max,6.000000e+08,6.000000e+08,41.774687,50.341059,113458.000000,20.000000



2.2 PRICE VS TOTAL_PRICE
Percentage of rows where price == total_price: 0.9998709997519226

Difference between price and total_price:


...

Properties with area > 10,000 m²: 389


,price,Sahə,area_unit,area_value,area_m2,location
45825,40000000.0,200000 sot,sot,200000.0,20000000.0,28 May m.
75857,3000.0,100000 sot,sot,100000.0,10000000.0,Fatmayı q.
21517,2500.0,100000 sot,sot,100000.0,10000000.0,Fatmayı q.
9746,2500.0,100000 sot,sot,100000.0,10000000.0,Fatmayı q.
4736,2500.0,100000 sot,sot,100000.0,10000000.0,Fatmayı q.
69308,26000000.0,60000 sot,sot,60000.0,6000000.0,Binəqədi r.
28995,19000000.0,60000 sot,sot,60000.0,6000000.0,Binəqədi r.
172,80000.0,7000 sot,sot,7000.0,700000.0,Maştağa q.
87428,2640000.0,4400 sot,sot,4400.0,440000.0,Qobu q.
37618,16000000.0,4000 sot,sot,4000.0,400000.0,Lökbatan q.



2.27 REMOVE TEMPORARY EDA VARIABLES

2.29 FINAL FEATURE CLEANUP
estate_rel_url_x vs estate_rel_url: 100.00%

2.30 FINAL DATASET CHECK
Final cleaned dataframe shape: (100775, 29)

Final columns:
['datetime_scrape_x', 'price', 'location', 'attributes', 'city_when', 'city', 'day_x', 'hour_x', 'products_label', 'description', 'owner_title', 'address', 'lat', 'lng', 'updated', 'views', 'day_y', 'Kateqoriya', 'Otaq sayı', 'Təmir', 'Çıxarış', 'İpoteka', 'estate_rel_url', 'extra_info', 'area_unit', 'area_value', 'area_m2', 'floor', 'total_floors']

Remaining missing values:


total_floors         24779
floor                24779
Otaq sayı             9412
owner_title            657
description            265
city_when                0
attributes               0
city                     0
day_x                    0
products_label           0
location                 0
price                    0
datetime_scrape_x        0
lat                      0
address                  0
hour_x                   0
lng                      0
day_y                    0
Kateqoriya               0
views                    0
updated                  0
Çıxarış                  0
Təmir                    0
İpoteka                  0
estate_rel_url           0
area_unit                0
extra_info               0
area_m2                  0
area_value               0
dtype: int64


Data types:
datetime_scrape_x     object
price                float64
location              object
attributes            object
city_when             object
city                  object
day_x                 object
hour_x                object
products_label        object
description           object
owner_title           object
address               object
lat                  float64
lng                  float64
updated               object
views                  int64
day_y                 object
Kateqoriya            object
Otaq sayı            float64
Təmir                 object
Çıxarış               object
İpoteka               object
estate_rel_url        object
extra_info            object
area_unit             object
area_value           float64
area_m2              float64
floor                float64
total_floors         float64
dtype: object

Remaining leakage/identifier columns: []

Unique property groups: 64454
Total observations: 100775

X shape: (100775, 28)
y shape: 